In [ ]:
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'
MINCUBESAMPLES = 100

SRFUNCTIONS = {
    'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
    'sqrt':np.sqrt,'exp':np.exp,'log':np.log,'abs':np.abs,
    'sin':np.sin,'cos':np.cos,'max':np.maximum,'min':np.minimum,
    '_safepow':lambda a,b:np.abs(a)**b}

import re
def _prepare_form(form):
    return re.sub(r'(\w+)\^(\w+)',r'_safepow(\1,\2)',form)

def eval_form(form,columns,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    ns.update(columns)
    ns.update(constants)
    out = eval(_prepare_form(form),ns)
    if np.ndim(out)==0:
        n = len(next(v for v in columns.values() if hasattr(v,'__len__')))
        out = np.full(n,float(out))
    return np.asarray(out,dtype=float)

In [4]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS['tp_mean']
STD  = STATS['tp_std']
ZMIN = (0.0 - MEAN) / STD

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds['dsig'].values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lfraw = flat('lf')
    shfraw = flat('shf')
    lhfraw = flat('lhf')
    blraw = flat('bl') if 'bl' in ds else np.zeros(ntime*ds.sizes['lat']*ds.sizes['lon'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsraw = ds['tp'].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)
meankernel = np.mean(kernels,axis=0)
weighted = fields * meankernel[None,:,:] * dsig[None,None,:]
if surfmask is not None:
    weighted = weighted * surfmask[:,None,:]
integrals = weighted.sum(axis=2)
rhraw,thetaeraw,thetaestarraw = integrals[:,0],integrals[:,1],integrals[:,2]

valid = np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw) & np.isfinite(obsraw)
rh,thetae,thetaestar = rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf,shf,lhf = lfraw[valid],shfraw[valid],lhfraw[valid]
bl = blraw[valid]
obs = obsraw[valid]
landmask  = lf > 0.5
oceanmask = lf < 0.5
print(f'Loaded {valid.sum():,} valid samples ({landmask.sum():,} land, {oceanmask.sum():,} ocean)')

Loaded 1,437,408 valid samples (428,352 land, 1,009,056 ocean)


In [5]:
regpath = os.path.join(MODELSDIR,'sr','optimized_equations.pkl')
with open(regpath,'rb') as f:
    REGISTRY = pickle.load(f)
SRMODELS = CONFIGS['experiments']['sr']['optimizedeqs']
ORDER = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}
COLORS = {name:SRMODELS[name]['color'] for name in ORDER}
print(f'Loaded {len(ORDER)} optimized equations: {[LABELS[n] for n in ORDER]}')

Loaded 4 optimized equations: ['SR-BL', 'SR-ATM', 'SR-SFC', 'SR-ALL']


In [6]:
def predict_eq(name,columns):
    entry = REGISTRY[name]
    form,constants = entry['form'],entry['constants']
    raw = eval_form(form,columns,constants)
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

def get_columns(**overrides):
    cols = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    cols.update(overrides)
    for eqname,entry in REGISTRY.items():
        if eqname in overrides:
            continue
        cols[eqname] = eval_form(entry['form'],cols,entry['constants'])
    return cols

In [7]:
BASEVARS = ['rh','thetae','thetaestar','lf','shf','lhf','bl']

def cube_monotonicity_test(name,target_var,expected_sign,other_vars,ncubes,mask=None):
    '''
    Grundner et al. (2024) cube-based monotonicity test.

    Divides the space of `other_vars` into ncubes^len(other_vars) hypercubes.
    Within each cube, fits a linear slope of P w.r.t. `target_var`.
    Returns the fraction of cubes where the slope has the expected sign.
    '''
    cols = get_columns()
    targetvals = cols[target_var]
    othervalslist = [cols[v] for v in other_vars]
    nother = len(other_vars)

    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)

    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]

    nsatisfied = 0
    ntested = 0

    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]

    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue

        nsel = sel.sum()
        x = targetvals[sel]
        overrides = {k:cols[k][sel] for k in BASEVARS}
        overrides[target_var] = x
        for v,binarr,edgearr in zip(other_vars,bins,edges):
            ci = cidx
            for j in range(nother-1,-1,-1):
                if other_vars[j] == v:
                    bi = ci % ncubes
                    break
                ci //= ncubes
            midpoint = 0.5 * (edgearr[bi] + edgearr[bi+1])
            overrides[v] = np.full(nsel,midpoint)

        cubecols = get_columns(**overrides)
        p = predict_eq(name,cubecols)

        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,p) / (np.dot(xc,xc) + 1e-12)

        ntested += 1
        if expected_sign >= 0 and slope >= 0:
            nsatisfied += 1
        elif expected_sign < 0 and slope <= 0:
            nsatisfied += 1

    return nsatisfied,ntested

In [8]:
CONSTRAINTS = {
    'PC3':{  
        'label':r'$\partial P/\partial \widehat{\mathrm{RH}} \geq 0$',
        'target_var':'rh',
        'expected_sign':1,
        'other_vars':['thetae','thetaestar']},
    'PC4':{
        'label':r'$\partial P/\partial \widehat{\theta_e} \geq 0$',
        'target_var':'thetae',
        'expected_sign':1,
        'other_vars':['rh','thetaestar']},
    'PC5':{
        'label':r'$\partial P/\partial \widehat{\theta_e^*} \leq 0$',
        'target_var':'thetaestar',
        'expected_sign':-1,
        'other_vars':['rh','thetae']}}

NCUBES = [3,4,5,6,7]
print(f'Testing {len(CONSTRAINTS)} constraints across {len(NCUBES)} cube sizes')

Testing 3 constraints across 5 cube sizes


In [ ]:
def data_cube_monotonicity_test(target_var,expected_sign,other_vars,ncubes,mask=None):
    featurevals = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    targetvals = featurevals[target_var]
    othervalslist = [featurevals[v] for v in other_vars]
    nother = len(other_vars)
    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]
    nsatisfied,ntested = 0,0
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]
    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        x,y = targetvals[sel],obs[sel]
        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,y) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expected_sign >= 0 and slope >= 0) or (expected_sign < 0 and slope <= 0):
            nsatisfied += 1
    return nsatisfied,ntested

dataresults = {}
for pcname,pc in CONSTRAINTS.items():
    dataresults[pcname] = {}
    for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
        pcts = []
        for n in NCUBES:
            sat,tot = data_cube_monotonicity_test(pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask=mask)
            pcts.append(sat/max(tot,1)*100)
        dataresults[pcname][region] = np.mean(pcts)

In [ ]:
modelresults = {}
for name in ORDER:
    modelresults[name] = {}
    for pcname,pc in CONSTRAINTS.items():
        modelresults[name][pcname] = {}
        for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
            pcts = []
            for n in NCUBES:
                sat,tot = cube_monotonicity_test(name,pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask=mask)
                pcts.append(sat/max(tot,1)*100)
            modelresults[name][pcname][region] = np.mean(pcts)

In [ ]:
PCLABELS = {k:v['label'] for k,v in CONSTRAINTS.items()}
REGIONS = ['Land','Ocean','All']

rows = []
for region in REGIONS:
    row = {'Region':region,'Source':'ERA5/IMERG'}
    for pcname in CONSTRAINTS:
        row[PCLABELS[pcname]] = dataresults[pcname][region]
    rows.append(row)
for name in ORDER:
    for region in REGIONS:
        row = {'Region':region,'Source':LABELS[name]}
        for pcname in CONSTRAINTS:
            row[PCLABELS[pcname]] = modelresults[name][pcname][region]
        rows.append(row)

df = pd.DataFrame(rows).set_index(['Source','Region'])
df = df.style.format('{:.1f}%').set_caption(
    f'Physical constraint satisfaction (%, averaged over N={NCUBES}, test split, min {MINCUBESAMPLES} samples/cube)')
df

## Ocean dP/dRH violations: real physics or data artifact?

PC3 ($\partial P / \partial \widehat{\text{RH}} \geq 0$) is satisfied ~99% over land but only ~60% over ocean in the data. Three diagnostics below:

1. **Where are the violations?** Map the sign of the local dP/dRH slope across the ocean domain to see if violations cluster geographically (suggesting a physical regime) or scatter randomly (suggesting noise).
2. **What thermodynamic regime do violations live in?** Compare the joint ($\hat{\theta}_e$, $\hat{\theta}_e^*$) distribution of violating vs. satisfying ocean cubes — if violations cluster in stable/dry regimes, they may reflect real suppression of convection despite high RH.
3. **How strong are the negative slopes?** Compare the magnitude of negative vs. positive dP/dRH slopes over ocean — weak negatives relative to strong positives suggest noise, while comparable magnitudes suggest a real signal.

In [ ]:
NCUBE = 5
pc = CONSTRAINTS['PC3']
target_var = pc['target_var']
other_vars = pc['other_vars']

featurevals = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
               'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
targetvals = featurevals[target_var]
othervalslist = [featurevals[v] for v in other_vars]
nother = len(other_vars)

cuberows = []
for region,mask in [('Land',landmask),('Ocean',oceanmask)]:
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),NCUBE+1) for v in othervalslist]
    bins = [np.clip(np.digitize(v,e)-1,0,NCUBE-1) for v,e in zip(othervalslist,edges)]
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * NCUBE + bins[i]
    for cidx in range(NCUBE**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        x,y = targetvals[sel],obs[sel]
        xc = x - x.mean()
        slope = np.dot(xc,y) / (np.dot(xc,xc) + 1e-12)
        cuberows.append({
            'region':region,
            'slope':slope,
            'satisfied':slope >= 0,
            'nsamples':int(sel.sum()),
            'rh_mean':rh[sel].mean(),
            'thetae_mean':thetae[sel].mean(),
            'thetaestar_mean':thetaestar[sel].mean(),
            'precip_mean':obs[sel].mean(),
            'stability':thetaestar[sel].mean() - thetae[sel].mean()})

cubedf = pd.DataFrame(cuberows)
ocean = cubedf[cubedf['region']=='Ocean']
land = cubedf[cubedf['region']=='Land']
print(f'Ocean cubes: {len(ocean)} ({ocean.satisfied.sum()} satisfy PC3, '
      f'{(~ocean.satisfied).sum()} violate)')
print(f'Land cubes:  {len(land)} ({land.satisfied.sum()} satisfy PC3, '
      f'{(~land.satisfied).sum()} violate)')

In [ ]:
fig,axs = pplt.subplots(ncols=3,figwidth=12,refheight=3.5,sharey=False)

# (a) Slope magnitude comparison: violating vs satisfying ocean cubes
ax = axs[0]
sat_slopes = ocean[ocean['satisfied']]['slope'].values
vio_slopes = ocean[~ocean['satisfied']]['slope'].values
bins_hist = np.linspace(min(ocean['slope'].min(),-0.5),ocean['slope'].max(),30)
ax.hist(sat_slopes,bins=bins_hist,color='#2355a1',alpha=0.7,label=f'Satisfied (n={len(sat_slopes)})')
ax.hist(vio_slopes,bins=bins_hist,color='#C44E52',alpha=0.7,label=f'Violated (n={len(vio_slopes)})')
ax.axvline(0,color='k',ls='--',lw=0.8)
ax.format(xlabel=r'$\partial P / \partial \widehat{\mathrm{RH}}$ slope',ylabel='Number of cubes',
          title='Slope magnitude')
ax.legend(loc='ur',fontsize=7)

# (b) Thermodynamic regime: stability vs RH for violating vs satisfying cubes
ax = axs[1]
ax.scatter(ocean[ocean['satisfied']]['rh_mean'],ocean[ocean['satisfied']]['stability'],
           c='#2355a1',s=20,alpha=0.6,label='Satisfied',zorder=2)
ax.scatter(ocean[~ocean['satisfied']]['rh_mean'],ocean[~ocean['satisfied']]['stability'],
           c='#C44E52',s=40,alpha=0.8,label='Violated',marker='x',zorder=3)
ax.format(xlabel=r'Cube mean $\widehat{\mathrm{RH}}$',
          ylabel=r'Cube mean $\widehat{\theta_e^*} - \widehat{\theta_e}$ (stability)',
          title='Thermodynamic regime')
ax.legend(loc='ur',fontsize=7)

# (c) Precipitation in violating vs satisfying cubes
ax = axs[2]
ax.scatter(ocean[ocean['satisfied']]['rh_mean'],ocean[ocean['satisfied']]['precip_mean'],
           c='#2355a1',s=20,alpha=0.6,label='Satisfied',zorder=2)
ax.scatter(ocean[~ocean['satisfied']]['rh_mean'],ocean[~ocean['satisfied']]['precip_mean'],
           c='#C44E52',s=40,alpha=0.8,label='Violated',marker='x',zorder=3)
ax.format(xlabel=r'Cube mean $\widehat{\mathrm{RH}}$',ylabel='Cube mean precip (mm)',
          title='Precipitation regime')
ax.legend(loc='ur',fontsize=7)

axs.format(abc=True,titleloc='l')
pplt.show()

In [ ]:
vio = ocean[~ocean['satisfied']]
sat = ocean[ocean['satisfied']]

summary = pd.DataFrame({
    'Metric':['Number of cubes',
              'Mean |slope|',
              r'Mean $\widehat{\mathrm{RH}}$',
              r'Mean stability ($\widehat{\theta_e^*} - \widehat{\theta_e}$)',
              'Mean precip (mm)',
              'Mean samples per cube'],
    'Satisfied':[len(sat),
                 f'{np.abs(sat.slope).mean():.4f}',
                 f'{sat.rh_mean.mean():.3f}',
                 f'{sat.stability.mean():.3f}',
                 f'{sat.precip_mean.mean():.2f}',
                 f'{sat.nsamples.mean():.0f}'],
    'Violated':[len(vio),
                f'{np.abs(vio.slope).mean():.4f}',
                f'{vio.rh_mean.mean():.3f}',
                f'{vio.stability.mean():.3f}',
                f'{vio.precip_mean.mean():.2f}',
                f'{vio.nsamples.mean():.0f}']
}).set_index('Metric')

print('Ocean dP/dRH cubes — satisfied vs. violated:')
summary